# Milestone 5

In [1]:
import torch
import torch.nn.functional as F
import pandas as pd
import numpy as np
from transformers import AutoTokenizer,AutoModelForSequenceClassification

CHOICES = ["A", "B", "C", "D", "E"]
LABEL2OPT = {0: "A", 1: "B", 2: "C", 3: "D", 4: "E"}
OPT2LABEL = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}
NUM_LABELS = 5
MAX_LEN = 512

W_DEBERTA = 0.70
W_ROBERTA = 0.30


TRAIN_CSV = "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
TEST_CSV = "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"

DEBERTA_CKPT = "deberta-v3-small-finetuned" 
ROBERTA_CKPT = "roberta-base-finetuned"  

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cuda


In [2]:
train_df = pd.read_csv(TRAIN_CSV)
test_df  = pd.read_csv(TEST_CSV)

for c in CHOICES:
    train_df[c] = train_df[c].fillna("").astype(str)
    test_df[c] = test_df[c].fillna("").astype(str)
train_df["prompt"] = train_df["prompt"].fillna("").astype(str)
test_df["prompt"] = test_df["prompt"].fillna("").astype(str)

print(f"Train: {len(train_df)} rows | Test: {len(test_df)} rows")
print(f"Test row index 25 prompt: {test_df.iloc[25]['prompt'][:100]}...")

Train: 2000 rows | Test: 2000 rows
Test row index 25 prompt: Choose the correct answer: What is Hesse's principle of transfer in geometry? carefully....


In [3]:
from torch.utils.data import Dataset, DataLoader
from transformers import get_linear_schedule_with_warmup

def format_input(row):
    prompt = str(row["prompt"])
    options = " [SEP] ".join([f"{c}: {str(row[c])}" for c in CHOICES])
    return f"{prompt} [SEP] {options}"


class MCQDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=512, has_labels=True):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.has_labels = has_labels

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = format_input(row)
        encoding = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )
        item = {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0)
        }
        if self.has_labels:
            item["labels"] = torch.tensor(OPT2LABEL[row["answer"]], dtype=torch.long)
        return item


def train_model(model_name, save_path, train_df, epochs=3, batch_size=8, lr=2e-5):
    print(f"\n{'='*60}")
    print(f"Fine-tuning: {model_name}")
    print(f"{'='*60}")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=NUM_LABELS
    ).to(device)

    dataset = MCQDataset(train_df, tokenizer, max_len=MAX_LEN, has_labels=True)
    loader  = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    total_steps = len(loader) * epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps
    )

    model.train()
    for epoch in range(epochs):
        total_loss = 0
        correct = 0
        total = 0
        for batch_idx, batch in enumerate(loader):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels)
            loss = outputs.loss
            logits = outputs.logits

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

            total_loss += loss.item()
            preds = torch.argmax(logits, dim=-1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

            if (batch_idx + 1) % 50 == 0:
                print(f"  Epoch {epoch+1} | Step {batch_idx+1}/{len(loader)} | "
                      f"Loss: {loss.item():.4f} | Acc: {correct/total:.4f}")

        avg_loss = total_loss / len(loader)
        accuracy = correct / total
        print(f"  Epoch {epoch+1}/{epochs} — Avg Loss: {avg_loss:.4f} | Accuracy: {accuracy:.4f}")

    model.save_pretrained(save_path)
    tokenizer.save_pretrained(save_path)
    print(f"\n  Model saved to: {save_path}")

    return model, tokenizer

In [4]:
deberta_model, deberta_tokenizer = train_model(model_name="microsoft/deberta-v3-small",save_path=DEBERTA_CKPT,
                                    train_df=train_df,epochs=3,batch_size=8,lr=2e-5)


Fine-tuning: microsoft/deberta-v3-small


config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/286M [00:00<?, ?B/s]

...

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.weight                       | MISSING    | 
classifier.bias         

model.safetensors:   0%|          | 0.00/286M [00:00<?, ?B/s]

  Epoch 1 | Step 50/250 | Loss: nan | Acc: 0.1900
  Epoch 1 | Step 100/250 | Loss: nan | Acc: 0.1787
  Epoch 1 | Step 150/250 | Loss: nan | Acc: 0.1925
  Epoch 1 | Step 200/250 | Loss: nan | Acc: 0.1875
  Epoch 1 | Step 250/250 | Loss: nan | Acc: 0.1845
  Epoch 1/3 — Avg Loss: nan | Accuracy: 0.1845
  Epoch 2 | Step 50/250 | Loss: nan | Acc: 0.1825
  Epoch 2 | Step 100/250 | Loss: nan | Acc: 0.1663
  Epoch 2 | Step 150/250 | Loss: nan | Acc: 0.1742
  Epoch 2 | Step 200/250 | Loss: nan | Acc: 0.1794
  Epoch 2 | Step 250/250 | Loss: nan | Acc: 0.1845
  Epoch 2/3 — Avg Loss: nan | Accuracy: 0.1845
  Epoch 3 | Step 50/250 | Loss: nan | Acc: 0.2075
  Epoch 3 | Step 100/250 | Loss: nan | Acc: 0.1850
  Epoch 3 | Step 150/250 | Loss: nan | Acc: 0.1850
  Epoch 3 | Step 200/250 | Loss: nan | Acc: 0.1869
  Epoch 3 | Step 250/250 | Loss: nan | Acc: 0.1845
  Epoch 3/3 — Avg Loss: nan | Accuracy: 0.1845


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


  Model saved to: deberta-v3-small-finetuned


In [5]:
roberta_model, roberta_tokenizer = train_model(
    model_name="roberta-base",
    save_path=ROBERTA_CKPT,
    train_df=train_df,
    epochs=3,
    batch_size=8,
    lr=2e-5
)


Fine-tuning: roberta-base


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  Epoch 1 | Step 50/250 | Loss: 1.5895 | Acc: 0.1875
  Epoch 1 | Step 100/250 | Loss: 1.6423 | Acc: 0.2075
  Epoch 1 | Step 150/250 | Loss: 1.5568 | Acc: 0.2367
  Epoch 1 | Step 200/250 | Loss: 1.1700 | Acc: 0.3262
  Epoch 1 | Step 250/250 | Loss: 0.4447 | Acc: 0.4330
  Epoch 1/3 — Avg Loss: 1.2895 | Accuracy: 0.4330
  Epoch 2 | Step 50/250 | Loss: 0.0211 | Acc: 0.9725
  Epoch 2 | Step 100/250 | Loss: 0.0168 | Acc: 0.9775
  Epoch 2 | Step 150/250 | Loss: 0.0049 | Acc: 0.9842
  Epoch 2 | Step 200/250 | Loss: 0.0035 | Acc: 0.9875
  Epoch 2 | Step 250/250 | Loss: 0.0029 | Acc: 0.9900
  Epoch 2/3 — Avg Loss: 0.0498 | Accuracy: 0.9900
  Epoch 3 | Step 50/250 | Loss: 0.0024 | Acc: 1.0000
  Epoch 3 | Step 100/250 | Loss: 0.0021 | Acc: 1.0000
  Epoch 3 | Step 150/250 | Loss: 0.0023 | Acc: 1.0000
  Epoch 3 | Step 200/250 | Loss: 0.0019 | Acc: 1.0000
  Epoch 3 | Step 250/250 | Loss: 0.0022 | Acc: 1.0000
  Epoch 3/3 — Avg Loss: 0.0023 | Accuracy: 1.0000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


  Model saved to: roberta-base-finetuned


## Load Fine-Tuned Models

In [6]:
deberta_tokenizer = AutoTokenizer.from_pretrained(DEBERTA_CKPT)
deberta_model = AutoModelForSequenceClassification.from_pretrained(
    DEBERTA_CKPT, num_labels=NUM_LABELS
).to(device)
deberta_model.eval()
print(f"DeBERTa loaded from: {DEBERTA_CKPT}")
print(f"  Parameters: {sum(p.numel() for p in deberta_model.parameters()):,}")

roberta_tokenizer = AutoTokenizer.from_pretrained(ROBERTA_CKPT)
roberta_model = AutoModelForSequenceClassification.from_pretrained(
    ROBERTA_CKPT, num_labels=NUM_LABELS
).to(device)
roberta_model.eval()
print(f"\nRoBERTa loaded from: {ROBERTA_CKPT}")
print(f"  Parameters: {sum(p.numel() for p in roberta_model.parameters()):}")

Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

DeBERTa loaded from: deberta-v3-small-finetuned
  Parameters: 141,898,757


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


RoBERTa loaded from: roberta-base-finetuned
  Parameters: 124649477


In [7]:
def get_probs(model, tokenizer, text, max_len=MAX_LEN):

    encoding = tokenizer(
        text,
        padding="max_length",
        truncation=True,
        max_length=max_len,
        return_tensors="pt")
    input_ids = encoding["input_ids"].to(device)
    attention_mask = encoding["attention_mask"].to(device)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        probs = F.softmax(logits, dim=-1).squeeze(0).cpu().numpy()

    return probs


def get_probs_batch(model, tokenizer, texts, max_len=MAX_LEN, batch_size=16):
    all_probs = []
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i : i + batch_size]
        encoding = tokenizer(
            batch_texts,
            padding="max_length",
            truncation=True,
            max_length=max_len,
            return_tensors="pt")
        input_ids = encoding["input_ids"].to(device)
        attention_mask = encoding["attention_mask"].to(device)

        with torch.no_grad():
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            probs = F.softmax(outputs.logits, dim=-1).cpu().numpy()  # [batch, 5]

        all_probs.append(probs)

    return np.concatenate(all_probs, axis=0)


def probs_to_top3(probs):
    ranked_indices = np.argsort(probs)[::-1]
    top3_letters = [LABEL2OPT[idx] for idx in ranked_indices[:3]]
    return " ".join(top3_letters)


def probs_to_top1(probs):
    return LABEL2OPT[np.argmax(probs)]

In [8]:
row_25 = test_df.iloc[25]
text_25 = format_input(row_25)

print(f"Row index 25 prompt: {row_25['prompt'][:100]}...")
print(f"Formatted input (first 150 chars): {text_25[:150]}...\n")

deberta_probs_25 = get_probs(deberta_model, deberta_tokenizer, text_25)

print("DeBERTa probabilities for row index 25:")
for i, c in enumerate(CHOICES):
    marker = " ← HIGHEST" if i == np.argmax(deberta_probs_25) else ""
    print(f"  {c}: {deberta_probs_25[i]:.6f}{marker}")

top1_letter = LABEL2OPT[np.argmax(deberta_probs_25)]
top1_prob   = np.max(deberta_probs_25)

print(f"Q1: {top1_letter}, {top1_prob:.4f}")

Row index 25 prompt: Choose the correct answer: What is Hesse's principle of transfer in geometry? carefully....
Formatted input (first 150 chars): Choose the correct answer: What is Hesse's principle of transfer in geometry? carefully. [SEP] A: Hesse's principle of transfer is a concept in biolog...

DeBERTa probabilities for row index 25:
  A: nan ← HIGHEST
  B: nan
  C: nan
  D: nan
  E: nan
Q1: A, nan


In [9]:
roberta_probs_25 = get_probs(roberta_model, roberta_tokenizer, text_25)

print("RoBERTa probabilities for row index 25:")
for i, c in enumerate(CHOICES):
    print(f"  {c}: {roberta_probs_25[i]:.6f}")

avg_probs_25 = (deberta_probs_25 + roberta_probs_25) / 2.0

print(f"\nSimple average probabilities:")
for i, c in enumerate(CHOICES):
    marker = " ← HIGHEST" if i == np.argmax(avg_probs_25) else ""
    print(f"  {c}: ({deberta_probs_25[i]:.4f} + {roberta_probs_25[i]:.4f}) / 2 = {avg_probs_25[i]:.6f}{marker}")

top1_avg = LABEL2OPT[np.argmax(avg_probs_25)]

print(f"Q2: {top1_avg}")

RoBERTa probabilities for row index 25:
  A: 0.000295
  B: 0.000332
  C: 0.000532
  D: 0.000227
  E: 0.998614

Simple average probabilities:
  A: (nan + 0.0003) / 2 = nan ← HIGHEST
  B: (nan + 0.0003) / 2 = nan
  C: (nan + 0.0005) / 2 = nan
  D: (nan + 0.0002) / 2 = nan
  E: (nan + 0.9986) / 2 = nan
Q2: A


In [10]:
weighted_probs_25 = W_DEBERTA * deberta_probs_25 + W_ROBERTA * roberta_probs_25

print(f"Weighted ensemble (DeBERTa={W_DEBERTA}, RoBERTa={W_ROBERTA}):")
for i, c in enumerate(CHOICES):
    marker = " ← HIGHEST" if i == np.argmax(weighted_probs_25) else ""
    d_part = W_DEBERTA * deberta_probs_25[i]
    r_part = W_ROBERTA * roberta_probs_25[i]
    print(f"  {c}: {d_part:.4f} + {r_part:.4f} = {weighted_probs_25[i]:.6f}{marker}")

top1_weighted = LABEL2OPT[np.argmax(weighted_probs_25)]

print(f"Q3: {top1_weighted}")

Weighted ensemble (DeBERTa=0.7, RoBERTa=0.3):
  A: nan + 0.0001 = nan ← HIGHEST
  B: nan + 0.0001 = nan
  C: nan + 0.0002 = nan
  D: nan + 0.0001 = nan
  E: nan + 0.2996 = nan
Q3: A


In [11]:
top3_str_25 = probs_to_top3(weighted_probs_25)

ranked_all = np.argsort(weighted_probs_25)[::-1]
print("Full ranking (weighted ensemble):")
for rank, idx in enumerate(ranked_all, 1):
    print(f"  Rank {rank}: {LABEL2OPT[idx]} (prob={weighted_probs_25[idx]:.6f})")

print(f"Q4: {top3_str_25}")

Full ranking (weighted ensemble):
  Rank 1: E (prob=nan)
  Rank 2: D (prob=nan)
  Rank 3: C (prob=nan)
  Rank 4: B (prob=nan)
  Rank 5: A (prob=nan)
Q4: E D C


In [12]:
all_test_texts = [format_input(test_df.iloc[i]) for i in range(len(test_df))]

print(f"Running DeBERTa inference on {len(all_test_texts)} test rows...")
deberta_probs_all = get_probs_batch(
    deberta_model, deberta_tokenizer, all_test_texts, batch_size=16
)
print(f"  DeBERTa done: shape = {deberta_probs_all.shape}")

print(f"Running RoBERTa inference on {len(all_test_texts)} test rows...")
roberta_probs_all = get_probs_batch(
    roberta_model, roberta_tokenizer, all_test_texts, batch_size=16
)
print(f"  RoBERTa done: shape = {roberta_probs_all.shape}")

weighted_probs_all = W_DEBERTA * deberta_probs_all + W_ROBERTA * roberta_probs_all

predictions = [probs_to_top3(weighted_probs_all[i]) for i in range(len(test_df))]

submission = pd.DataFrame({
    "id": test_df["id"],
    "prediction": predictions,
})

# Save
submission.to_csv("submission.csv", index=False)

print(f"\nSubmission preview:")
print(submission.head(10))
print(f"Q5: {len(submission)} prediction rows")

Running DeBERTa inference on 2000 test rows...
  DeBERTa done: shape = (2000, 5)
Running RoBERTa inference on 2000 test rows...
  RoBERTa done: shape = (2000, 5)

Submission preview:
   id prediction
0   1      E D C
1   2      E D C
2   3      E D C
3   4      E D C
4   5      E D C
5   6      E D C
6   7      E D C
7   8      E D C
8   9      E D C
9  10      E D C
Q5: 2000 prediction rows


In [13]:
TTA_PREFIX = "Answer the following multiple-choice question carefully: "
N_TTA = 50

tta_different_count = 0
tta_details = []

for i in range(N_TTA):
    row = test_df.iloc[i]
    text_original = format_input(row)
    probs_original = get_probs(deberta_model, deberta_tokenizer, text_original)

    row_augmented = row.copy()
    row_augmented["prompt"] = TTA_PREFIX + str(row["prompt"])
    text_augmented = format_input(row_augmented)
    probs_augmented = get_probs(deberta_model, deberta_tokenizer, text_augmented)

    probs_tta = (probs_original + probs_augmented) / 2.0

    top1_original = probs_to_top1(probs_original)
    top1_tta = probs_to_top1(probs_tta)

    is_different = top1_original != top1_tta
    if is_different:
        tta_different_count += 1
        tta_details.append((i, top1_original, top1_tta))

print(f"TTA results (first {N_TTA} rows):")
print(f"  Rows where Top-1 changed: {tta_different_count}")
if tta_details:
    print(f"\n  Changed rows:")
    for idx, orig, tta in tta_details:
        print(f"    Row {idx}: {orig} → {tta}")

print(f"Q6: {tta_different_count}")

TTA results (first 50 rows):
  Rows where Top-1 changed: 0
Q6: 0


In [14]:
N_COMPARE = 100
deberta_probs_100  = deberta_probs_all[:N_COMPARE]
weighted_probs_100 = weighted_probs_all[:N_COMPARE]

disagreement_count = 0
disagreement_details = []

for i in range(N_COMPARE):
    top1_deberta  = probs_to_top1(deberta_probs_100[i])
    top1_ensemble = probs_to_top1(weighted_probs_100[i])

    if top1_deberta != top1_ensemble:
        disagreement_count += 1
        disagreement_details.append((i, top1_deberta, top1_ensemble))

print(f"DeBERTa vs Weighted Ensemble — Top-1 comparison (first {N_COMPARE} rows):")
print(f"  Rows with different Top-1: {disagreement_count}")
if disagreement_details:
    print(f"\n  Disagreement details:")
    for idx, d, e in disagreement_details[:20]:
        print(f"    Row {idx:3d}: DeBERTa={d}  Ensemble={e}")
    if len(disagreement_details) > 20:
        print(f"    ... and {len(disagreement_details) - 20} more")

print(f"Q7: {disagreement_count}")

DeBERTa vs Weighted Ensemble — Top-1 comparison (first 100 rows):
  Rows with different Top-1: 0
Q7: 0


In [15]:
positive_gain_count = 0
confidence_gains = []

for i in range(N_COMPARE):
    deberta_conf  = np.max(deberta_probs_100[i])
    ensemble_conf = np.max(weighted_probs_100[i])
    gain = ensemble_conf - deberta_conf
    confidence_gains.append(gain)

    if gain > 0:
        positive_gain_count += 1

gains_arr = np.array(confidence_gains)
print(f"Confidence gain statistics (first {N_COMPARE} rows):")
print(f"  Mean gain:     {gains_arr.mean():.6f}")
print(f"  Median gain:   {np.median(gains_arr):.6f}")
print(f"  Min gain:      {gains_arr.min():.6f}")
print(f"  Max gain:      {gains_arr.max():.6f}")
print(f"  Positive:      {positive_gain_count}")
print(f"  Zero/Negative: {N_COMPARE - positive_gain_count}")

print(f"Q8: {positive_gain_count}")

Confidence gain statistics (first 100 rows):
  Mean gain:     nan
  Median gain:   nan
  Min gain:      nan
  Max gain:      nan
  Positive:      0
  Zero/Negative: 100
Q8: 0


In [16]:
top3_changed_count = 0
top3_change_details = []

for i in range(N_COMPARE):
    top3_deberta  = probs_to_top3(deberta_probs_100[i])
    top3_ensemble = probs_to_top3(weighted_probs_100[i])

    if top3_deberta != top3_ensemble:
        top3_changed_count += 1
        top3_change_details.append((i, top3_deberta, top3_ensemble))

print(f"Top-3 ranking comparison (first {N_COMPARE} rows):")
print(f"  Rows with changed Top-3: {top3_changed_count}")
if top3_change_details:
    print(f"\n  Changed rows (showing first 20):")
    for idx, d, e in top3_change_details[:20]:
        print(f"    Row {idx:3d}: DeBERTa=[{d}]  Ensemble=[{e}]")
    if len(top3_change_details) > 20:
        print(f"    ... and {len(top3_change_details) - 20} more")

print(f"Q9: {top3_changed_count}")

Top-3 ranking comparison (first 100 rows):
  Rows with changed Top-3: 0
Q9: 0


In [17]:
N_VAL = 100

def map_at_3_score(true_label_idx, probs):
    ranked = np.argsort(probs)[::-1][:3]  # top 3 indices
    for rank, pred_idx in enumerate(ranked):
        if pred_idx == true_label_idx:
            return 1.0 / (rank + 1)
    return 0.0


val_texts = [format_input(train_df.iloc[i]) for i in range(N_VAL)]
val_labels = [OPT2LABEL[train_df.iloc[i]["answer"]] for i in range(N_VAL)]

print(f"Running DeBERTa inference on {N_VAL} validation samples...")
deberta_probs_val = get_probs_batch(
    deberta_model, deberta_tokenizer, val_texts, batch_size=16
)

print(f"Running RoBERTa inference on {N_VAL} validation samples...")
roberta_probs_val = get_probs_batch(
    roberta_model, roberta_tokenizer, val_texts, batch_size=16
)

weighted_probs_val = W_DEBERTA * deberta_probs_val + W_ROBERTA * roberta_probs_val

ap_scores = []
for i in range(N_VAL):
    ap = map_at_3_score(val_labels[i], weighted_probs_val[i])
    ap_scores.append(ap)

map3 = np.mean(ap_scores)

print(f"\nMAP@3 Breakdown:")
print(f"  Correct at Rank 1: {sum(1 for s in ap_scores if s == 1.0)}")
print(f"  Correct at Rank 2: {sum(1 for s in ap_scores if s == 0.5)}")
print(f"  Correct at Rank 3: {sum(1 for s in ap_scores if abs(s - 1/3) < 0.01)}")
print(f"  Not in Top 3:      {sum(1 for s in ap_scores if s == 0.0)}")

print(f"Q10: MAP@3 = {map3:.4f}")

Running DeBERTa inference on 100 validation samples...
Running RoBERTa inference on 100 validation samples...

MAP@3 Breakdown:
  Correct at Rank 1: 16
  Correct at Rank 2: 14
  Correct at Rank 3: 23
  Not in Top 3:      47
Q10: MAP@3 = 0.3067
